In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.sparse import issparse, csr_matrix
import scanpy as sc
import os

from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_processed = os.path.join(base_path, "Data/processed")
monocle_dir    = os.path.join(base_path, "monocle_input")
os.makedirs(monocle_dir, exist_ok=True)

adata = sc.read_h5ad(os.path.join(data_processed, "adata_integrated.h5ad"))

print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"obs columns: {list(adata.obs.columns)}")

# 1. Expression matrix as Matrix Market — sparse, readable by R's Matrix package
X = adata.X if issparse(adata.X) else csr_matrix(adata.X)
sio.mmwrite(os.path.join(monocle_dir, "expression_matrix.mtx"), X)
print(f"Expression matrix: {X.shape[0]:,} genes × {X.shape[1]:,} cells")

# 2. Cell barcodes and gene names
pd.Series(adata.obs_names).to_csv(
    os.path.join(monocle_dir, "cell_barcodes.csv"), index=False, header=False
)
pd.Series(adata.var_names).to_csv(
    os.path.join(monocle_dir, "gene_names.csv"), index=False, header=False
)

# 3. Cell metadata — only columns needed for trajectory analysis
meta_cols = ['dataset', 'treatment', 'timepoint_str', 'timepoint_hours', 'leiden_scvi']
meta_cols = [c for c in meta_cols if c in adata.obs.columns]
adata.obs[meta_cols].to_csv(os.path.join(monocle_dir, "cell_metadata.csv"))

# 4. scVI latent space (36,067 × 30)
n_latent = adata.obsm['X_scvi'].shape[1]
pd.DataFrame(
    adata.obsm['X_scvi'],
    index=adata.obs_names,
    columns=[f'scVI_{i+1}' for i in range(n_latent)]
).to_csv(os.path.join(monocle_dir, "scvi_latent.csv"))

print(f"\nExport complete → {monocle_dir}")
print(f"  expression_matrix.mtx: {X.shape}")
print(f"  cell_barcodes.csv:     {adata.n_obs:,} cells")
print(f"  gene_names.csv:        {adata.n_vars:,} genes")
print(f"  cell_metadata.csv:     {len(meta_cols)} columns")
print(f"  scvi_latent.csv:       {adata.obsm['X_scvi'].shape}")